# 03 — ID drift audit

行IDが単なる識別子か、データ生成順序を表すproxyかを確認します。
IDはモデルには入れませんが、ID方向で目的変数やカテゴリ構成が変化していると
random CVがtestを正しく模倣しない可能性があります。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from config import Baseline

TARGET = Baseline.TARGET
ID_COLUMN = Baseline.ID_COLUMN
train = pd.read_csv(ROOT / "input" / "train.csv")
test = pd.read_csv(ROOT / "input" / "test.csv")
print("train ID:", train[ID_COLUMN].min(), "to", train[ID_COLUMN].max())
print("test ID :", test[ID_COLUMN].min(), "to", test[ID_COLUMN].max())

## ID区間ごとの解約率

20個の等頻度binで、件数と解約率のトレンドを確認します。

In [ ]:
audit = train[[ID_COLUMN, TARGET, "Contract"]].copy()
audit["target"] = audit[TARGET].eq("Yes").astype(int)
audit["id_bin"] = pd.qcut(audit[ID_COLUMN], 20, duplicates="drop")

id_summary = audit.groupby("id_bin", observed=True).agg(
    n=("target", "size"), churn_rate=("target", "mean")
)
display(id_summary)
id_summary["churn_rate"].plot(marker="o", title="Churn rate by ID quantile")
plt.ylabel("Churn rate")
plt.show()

## ID区間ごとの契約構成

解約率が動く場合、それが契約タイプなど既知特徴量の構成変化で説明できるか確認します。

In [ ]:
contract_mix = pd.crosstab(
    audit["id_bin"], audit["Contract"], normalize="index"
)
display(contract_mix)
contract_mix.plot(figsize=(10, 4), title="Contract mix by ID quantile")
plt.ylabel("Rate within ID bin")
plt.show()

## 解釈上の注意

IDと目的変数の関係が見えても、ID自体を特徴量へ追加するのは最後の手段です。
testのID範囲がtrainの外側なら、木モデルが学習範囲外へ一般化できず、CVだけが高くなることがあります。
まずは時間・顧客・生成バッチなど、IDが表す実体を特定します。